In [1]:
###########################################################
# Imports
###########################################################

import os
import shutil
import pathlib

import cv2
import numpy as np

from PIL import Image
from fastai.vision.all import *

###########################################################
# Function required by exported FastAI model
###########################################################

def label_func(fn):
    return path/"labels"/f"{fn.stem}_P{fn.suffix}"

###########################################################
# Load segmentation model once
###########################################################

if "learn" not in globals():

    # Required because the model was exported using Posix paths
    original_posix_path = pathlib.PosixPath
    pathlib.PosixPath = pathlib.WindowsPath

    learn = load_learner(fastaimodelpath)

    pathlib.PosixPath = original_posix_path

    print("Loaded segmentation model:")
    print(fastaimodelpath)

NameError: name 'fastaimodelpath' is not defined

In [ ]:
###########################################################
# Load current image
###########################################################
imagename = sorted_file_list[imagenumber]
image_path = rawimagesetdir + imagename

print("Processing image:")
print(imagename)
image = Image.open(image_path)
width, height = image.size

In [ ]:
###########################################################
# Clear temporary processing folders
###########################################################

cropdump_directory = mainpipelocation + "cropdump/"
cropdumpseg_directory = mainpipelocation + "cropdumpseg/"
masksdump_directory = mainpipelocation + "masksdump/"
yolo_runs_directory = here + "yolov5/runs/"

temporary_directories = [
    cropdump_directory,
    cropdumpseg_directory,
    masksdump_directory,
    yolo_runs_directory,
]

for directory in temporary_directories:

    try:
        shutil.rmtree(directory)

    except FileNotFoundError:
        pass

    os.makedirs(directory, exist_ok=True)

In [ ]:
###########################################################
# Create YOLO crops
###########################################################

yolo_crop_size = 640
yolo_crop_step = 320

number_yolo_columns = int(np.ceil((width - yolo_crop_size) / yolo_crop_step)) + 1
number_yolo_rows = int(np.ceil((height - yolo_crop_size) / yolo_crop_step)) + 1

crop_number = 1

for column in range(number_yolo_columns):

    for row in range(number_yolo_rows):

        x_start = column * yolo_crop_step
        y_start = row * yolo_crop_step

        crop_box = (
            x_start,
            y_start,
            x_start + yolo_crop_size,
            y_start + yolo_crop_size,
        )

        cropped_image = image.crop(crop_box)

        crop_filename = "cropimage" + str(crop_number) + ".png"

        cropped_image.save(
            cropdump_directory + crop_filename
        )

        crop_number = crop_number + 1

In [ ]:
###########################################################
# Run YOLO bubble detection
###########################################################

from ultralytics import YOLO

###########################################################
# Load YOLO model
###########################################################

if "yolo_model" not in globals():

    yolo_model = YOLO(yolomodelpath)

    print("Loaded YOLO model:")
    print(yolomodelpath)

###########################################################
# Run detection on cropped images
###########################################################

yolo_detections = []

crop_number = 1

while crop_number <= number_yolo_columns * number_yolo_rows:

    ###########################################################
    # Load crop
    ###########################################################

    crop_filename = cropdump_directory + "cropimage" + str(crop_number) + ".png"

    cropped_image = Image.open(crop_filename)

    ###########################################################
    # Run YOLO
    ###########################################################

    results = yolo_model(
        cropped_image,
        verbose=False
    )

    ###########################################################
    # Store detections
    ###########################################################

    crop_detections = []

    for result in results:

        for detection in result.boxes.data.tolist():

            x1, y1, x2, y2, score, class_id = detection

            crop_detections.append([
                x1,
                y1,
                x2,
                y2,
                score,
                class_id
            ])

    yolo_detections.append(crop_detections)

    crop_number = crop_number + 1

In [ ]:
###########################################################
# Create segmentation crops
###########################################################

segmentation_crop_size = 512

number_segmentation_columns = int(np.ceil(width / segmentation_crop_size))
number_segmentation_rows = int(np.ceil(height / segmentation_crop_size))

crop_number = 1

for column in range(number_segmentation_columns):

    for row in range(number_segmentation_rows):

        x_start = column * segmentation_crop_size
        y_start = row * segmentation_crop_size

        crop_box = (
            x_start,
            y_start,
            x_start + segmentation_crop_size,
            y_start + segmentation_crop_size,
        )

        cropped_image = image.crop(crop_box)

        crop_filename = "cropimage_seg" + str(crop_number) + ".png"

        cropped_image.save(
            cropdumpseg_directory + crop_filename
        )

        crop_number = crop_number + 1

In [ ]:
###########################################################
# Segment each crop
###########################################################

numberofcrops = number_segmentation_columns * number_segmentation_rows

crop_number = 1

while crop_number <= numberofcrops:

    ###########################################################
    # Load crop
    ###########################################################

    crop_filename = (
        cropdumpseg_directory
        + "cropimage_seg"
        + str(crop_number)
        + ".png"
    )

    crop_image = PILImage.create(crop_filename)

    ###########################################################
    # Run segmentation model
    ###########################################################

    test_dl = learn.dls.test_dl([crop_image])

    predictions = learn.get_preds(dl=test_dl)

    segmentation_mask = predictions[0][0].argmax(dim=0)

    ###########################################################
    # Save segmentation mask
    ###########################################################

    segmentation_mask_array = np.array(
        segmentation_mask.byte()
    )

    segmentation_mask_image = Image.fromarray(
        segmentation_mask_array
    )

    mask_filename = (
        masksdump_directory
        + "mask"
        + str(crop_number)
        + ".png"
    )

    segmentation_mask_image.save(mask_filename)

    print(crop_number / numberofcrops)

    crop_number = crop_number + 1

In [ ]:
###########################################################
# Reconstruct full segmentation mask
###########################################################

reconstruction_height = number_segmentation_rows * segmentation_crop_size
reconstruction_width = number_segmentation_columns * segmentation_crop_size

reconstruction = np.zeros(
    (
        reconstruction_height,
        reconstruction_width
    ),
    dtype=np.uint8
)

crop_number = 1

for column in range(number_segmentation_columns):

    for row in range(number_segmentation_rows):

        ###########################################################
        # Load segmentation crop
        ###########################################################

        mask_filename = (
            masksdump_directory
            + "mask"
            + str(crop_number)
            + ".png"
        )

        segment = cv2.imread(
            mask_filename,
            cv2.IMREAD_UNCHANGED
        )

        # Remove any unnecessary single channel dimension
        segment = np.squeeze(segment)

        ###########################################################
        # Position crop in reconstructed image
        ###########################################################

        y_start = row * segmentation_crop_size
        y_end = y_start + segmentation_crop_size

        x_start = column * segmentation_crop_size
        x_end = x_start + segmentation_crop_size

        ###########################################################
        # Add crop to reconstruction
        ###########################################################

        reconstruction[
            y_start:y_end,
            x_start:x_end
        ] = segment * 50

        crop_number = crop_number + 1

###########################################################
# Crop reconstruction back to original image size
###########################################################

segmentedimage = reconstruction[
    0:height,
    0:width
]

###########################################################
# Convert to 3-channel image for bubble masking
###########################################################

imagedraw = cv2.cvtColor(
    segmentedimage,
    cv2.COLOR_GRAY2BGR
)

In [ ]:
###########################################################
# Add YOLO bubble detections to segmentation mask
###########################################################

crop_number = 1

for column in range(number_yolo_columns):

    for row in range(number_yolo_rows):

        ###########################################################
        # Position of this crop in the full image
        ###########################################################

        xshift = column * yolo_crop_step
        yshift = row * yolo_crop_step

        ###########################################################
        # Get detections for this crop
        ###########################################################

        crop_detections = yolo_detections[crop_number - 1]

        ###########################################################
        # Draw detections
        ###########################################################

        for detection in crop_detections:

            x1, y1, x2, y2, score, class_id = detection

            ###########################################################
            # Convert crop coordinates to full-image coordinates
            ###########################################################

            X1 = int(x1 + xshift)
            Y1 = int(y1 + yshift)

            X2 = int(x2 + xshift)
            Y2 = int(y2 + yshift)

            ###########################################################
            # Mark bubble region
            ###########################################################

            cv2.rectangle(
                imagedraw,
                (X1, Y1),
                (X2, Y2),
                (0, 255, 0),
                -1
            )

        crop_number = crop_number + 1

In [ ]:
###########################################################
# Save final DOOM mask
###########################################################

maskfilename = imagename.replace(
    "image",
    "mask"
)

postbubblemaskname = (
    mainpipelocation
    + "fullmaskdump/"
    + maskfilename
)

final_mask = Image.fromarray(
    cv2.cvtColor(
        imagedraw,
        cv2.COLOR_BGR2RGB
    )
)

final_mask.save(
    postbubblemaskname
)

print("Saved:")
print(postbubblemaskname)